# Dark Web Threat Intelligence Toolkit — Demo Walkthrough

End-to-end demonstration of every module:  
**Scrape → Clean → Extract IOCs → Enrich → Classify → Analyse → Export → Summarise**

> Run cells top-to-bottom. The entire walkthrough completes in < 60 s on a modern laptop.

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure the project root is on the path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Use an isolated demo database so we don't pollute the main one
os.environ["DATABASE_PATH"] = str(PROJECT_ROOT / "data" / "demo_notebook.db")

from config import settings, setup_logging
logger = setup_logging("notebook")
print(f"Project root : {PROJECT_ROOT}")
print(f"Database     : {settings.get('project.database_path')}")
print(f"Python       : {sys.version.split()[0]}")

## 1. Scraping

Three scrapers ship with the toolkit:
| Class | What it fetches |
|---|---|
| `PasteScraper` | Pastebin-style fixtures / live paste sites |
| `SimulatedMarketScraper` | Realistic darknet-market HTML fixtures |
| `SeleniumScraper` | Headless-Chrome scraping (JS-rendered pages) |

In [ ]:
from scraper import PasteScraper, SimulatedMarketScraper

paste_scraper  = PasteScraper()
market_scraper = SimulatedMarketScraper()

paste_items  = paste_scraper.scrape(source="fixture")
market_items = market_scraper.scrape(fixture="all")
raw_items    = paste_items + market_items

print(f"Paste items  : {len(paste_items)}")
print(f"Market items : {len(market_items)}")
print(f"Total raw    : {len(raw_items)}")
print()
# Preview one item
sample = raw_items[0]
print(f"Source : {sample.source_name}")
print(f"Title  : {sample.title}")
print(f"Content preview: {sample.content[:200]}...")

## 2. Data Cleaning

In [ ]:
from pipeline.cleaner import TextCleaner

cleaner = TextCleaner()
cleaned = [cleaner.clean(item) for item in raw_items]

before = raw_items[0].content
after  = cleaned[0].content
print(f"Before ({len(before)} chars): {before[:150]}")
print()
print(f"After  ({len(after)} chars) : {after[:150]}")

## 3. IOC / Entity Extraction

The `EntityExtractor` uses **23 regex patterns** to pull structured IOCs from text:  
IPv4, IPv6, domains, URLs, CVEs, MD5/SHA-1/SHA-256 hashes, email addresses,  
Bitcoin/Monero wallets, MITRE ATT&CK technique IDs, PGP fingerprints, and more.

In [ ]:
from pipeline.entity_extractor import EntityExtractor

extractor = EntityExtractor(use_ner=False)   # NER disabled for speed; set True for spaCy
all_entities = []
for post in cleaned:
    entities = extractor.extract(post)
    all_entities.extend(entities)

# Summarise by type
from collections import Counter
type_counts = Counter(e.entity_type for e in all_entities)
print(f"Total IOCs extracted: {len(all_entities)}\n")
for etype, count in type_counts.most_common():
    print(f"  {etype:<30} {count}")

In [ ]:
# Inspect a few representative IOCs
import pandas as pd

rows = [
    {"type": e.entity_type, "value": e.value, "confidence": e.confidence}
    for e in all_entities[:20]
]
pd.DataFrame(rows)

## 4. Pipeline — Store to SQLite

In [ ]:
from pipeline import Pipeline

pipe = Pipeline(skip_enrichment=True, use_ner=False)
stats = pipe.run(cleaned)
pipe.close()

print(stats.to_dict())

## 5. Classification

Three classification layers, all returning the same output schema:
```python
{"category": str, "confidence": float, "scores": dict, "model": str}
```

| Layer | Speed | Requires training? |
|---|---|---|
| `KeywordClassifier` | < 1 ms/post | No |
| `MLClassifier` | ~ 1 ms/post (after train) | Yes (2 500 synthetic samples) |
| `TransformerClassifier` | ~ 100 ms/post | No (zero-shot BART-MNLI) |

In [ ]:
from classifier.keyword_classifier import KeywordClassifier

kw_clf = KeywordClassifier()

sample_texts = [
    "Ransomware group publishes 50 GB of exfiltrated hospital data on leak site.",
    "CVE-2024-1337 PoC exploit for Apache Struts RCE vulnerability released.",
    "Dumps of 2M card numbers with full track data — $5 each.",
    "APT29 C2 infrastructure using Cobalt Strike beacons on 185.220.101.5.",
]

results = [kw_clf.classify(t) for t in sample_texts]

for text, res in zip(sample_texts, results):
    print(f"{res['category']:<30} ({res['confidence']:.2f})  {text[:60]}")

In [ ]:
from classifier.ml_classifier import MLClassifier
from classifier.synthetic_data_generator import generate_synthetic_data

ml_clf = MLClassifier()

print("Generating 2 500 synthetic training samples...")
synth = generate_synthetic_data(num_samples=2500)
texts  = [s["content"]  for s in synth]
labels = [s["category"] for s in synth]

print("Training ML models (Random Forest, Gradient Boosting, Logistic Regression)...")
train_results = ml_clf.train(texts, labels, tune=False)

for name, res in train_results.items():
    print(f"  {name:<25} accuracy={res['accuracy']:.4f}  f1={res['f1']:.4f}")

In [ ]:
# ML model comparison report
report = ml_clf.get_comparison_report()
print(f"Best model: {report.get('best_model')}\n")

# Classify the same sample texts with the best ML model
ml_results = [ml_clf.classify(t) for t in sample_texts]
for text, res in zip(sample_texts, ml_results):
    print(f"{res['category']:<30} ({res['confidence']:.2f})  model={res['model']}")

## 6. MITRE ATT&CK Enrichment

In [ ]:
from classifier.mitre_mapper import MitreMapper

mapper = MitreMapper()

result = kw_clf.classify(sample_texts[0])
enriched = mapper.enrich_classification(result)

print(f"Category  : {enriched['category']}")
print(f"Techniques: {enriched.get('mitre_technique_ids', [])}")
print(f"Tactics   : {enriched.get('mitre_tactics', [])}")

## 7. Trend Analysis & Anomaly Detection

In [ ]:
from pipeline.db_loader import DatabaseLoader
from analysis.trend_analyzer import TrendAnalyzer
from analysis.anomaly_detector import AnomalyDetector

db       = DatabaseLoader()
db.init_schema()
analyzer = TrendAnalyzer(db)
detector = AnomalyDetector(db)

stats = analyzer.get_summary_stats()
print("Summary stats:")
for k, v in stats.items():
    print(f"  {k:<35} {v}")

In [ ]:
cat_dist     = analyzer.get_category_distribution("30d")
top_cves     = analyzer.get_top_cves(top_n=5)
keywords     = analyzer.get_trending_keywords("30d")
ioc_dist     = analyzer.get_ioc_distribution()
anomalies    = detector.detect(window_days=30)

print(f"Category distribution: {cat_dist['categories']}")
print(f"Top CVEs: {[c['cve_id'] for c in top_cves]}")
print(f"Trending keywords: {[k['keyword'] for k in keywords[:10]]}")
print(f"IOC type distribution: {ioc_dist}")
print(f"Anomalies detected: {len(anomalies)}")

db.close()

## 8. Report Generation

In [ ]:
from analysis.report_generator import ReportGenerator

gen = ReportGenerator()

report_args = dict(
    summary_stats=stats,
    category_dist=cat_dist,
    top_cves=top_cves,
    trending_keywords=keywords,
    ioc_dist=ioc_dist,
    anomalies=anomalies,
    actor_patterns=[],
)

md_path   = gen.generate_markdown(**report_args)
html_path = gen.generate_html(**report_args)

print(f"Markdown report : {md_path}")
print(f"HTML report     : {html_path}")

## 9. IOC Export — STIX 2.1, CSV, MISP

In [ ]:
from pipeline.db_loader import DatabaseLoader
from export.stix_exporter import StixExporter
from export.csv_exporter  import CsvExporter
from export.misp_exporter import MispExporter

db = DatabaseLoader()
db.init_schema()
entities        = db.get_entities(limit=5000)
cves            = db.get_cve_enrichments()
classifications = db.get_classifications(limit=2000)
db.close()

print(f"Entities to export: {len(entities)}")
print(f"CVE enrichments   : {len(cves)}")
print(f"Classifications   : {len(classifications)}")

In [ ]:
import json

out_dir = PROJECT_ROOT / "data" / "exports_notebook"

# STIX 2.1 bundle
stix_exp  = StixExporter()
stix_exp._output_dir = out_dir
stix_path = stix_exp.export(entities, cves, classifications)
bundle    = json.loads(Path(stix_path).read_text())
print(f"STIX bundle : {stix_path}")
print(f"  Objects   : {len(bundle['objects'])}")
print(f"  Types     : {set(o['type'] for o in bundle['objects'])}")

In [ ]:
# CSV export
csv_exp   = CsvExporter()
csv_exp._output_dir = out_dir
csv_paths = csv_exp.export_all(entities, classifications, cves)
print("CSV files:")
for p in csv_paths:
    print(f"  {p}")

In [ ]:
# MISP event
misp_exp  = MispExporter()
misp_exp._output_dir = out_dir
misp_path = misp_exp.export(entities, classifications)
misp_evt  = json.loads(Path(misp_path).read_text())
print(f"MISP event  : {misp_path}")
print(f"  Attributes: {len(misp_evt.get('Event', {}).get('Attribute', []))}")

## 10. AI Summarisation (local fallback)

In [ ]:
from ai_summarizer import ThreatSummarizer

# 'local' uses facebook/bart-large-cnn — no API key needed.
# Replace with 'openai' or 'anthropic' when API keys are in .env.
try:
    summarizer = ThreatSummarizer(backend="local")
    summary = summarizer.summarize(
        mode="executive",
        time_period="24h",
        total_posts=str(stats["total_posts"]),
        category_breakdown="ransomware: 3, data_breach: 2",
        critical_cves="CVE-2024-1337 (CVSS 9.8)",
        ioc_summary="12 IPs, 5 domains, 8 hashes",
        trends="Ransomware activity up 30% over 7-day baseline",
        cve_details="CVE-2024-1337 (CVSS 9.8)",
        mitre_techniques="T1486, T1059",
        ioc_details="12 IPs, 5 domains",
        ioc_data="12 IPs, 5 domains",
        anomalies="1 spike detected on 2024-01-15",
        categories="ransomware, data_breach",
    )
    print(summary)
except ImportError as exc:
    print(f"Local backend not available (transformers not installed): {exc}")
    print("Install with: pip install transformers torch")

## 11. DistilBERT Classifier (structural demo)

The `DistilBertClassifier` supports fine-tuning on your own labeled data.  
Here we demonstrate the API surface — actual fine-tuning requires `RUN_FINE_TUNE_TESTS=1` and takes ~5 min on GPU.

In [ ]:
from classifier import DistilBertClassifier

dbc = DistilBertClassifier()
print(f"Model name    : {dbc._model_name}")
print(f"Max length    : {dbc._max_length}")
print(f"Checkpoint dir: {dbc._checkpoint_dir}")
print()

# Demonstrates that classify() raises a clear error before fine_tune()
try:
    dbc.classify("test")
except RuntimeError as exc:
    print(f"Expected error before fine_tune(): {exc}")

## 12. CLI — Machine-Readable JSON Output

Every major command accepts `--json` for pipeline integration:

In [ ]:
import subprocess, json

def run_cli(*args):
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "cli.py"), *args],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    return json.loads(result.stdout)

scrape_result  = run_cli("scrape", "--source", "fixtures", "--json")
export_result  = run_cli("export", "--format", "stix",     "--json")

print("scrape --json :", scrape_result)
print("export --json :", export_result)

## Summary

| Module | Key classes | Outputs |
|---|---|---|
| `scraper` | PasteScraper, FeedScraper, SeleniumScraper | `RawPost` objects |
| `pipeline` | TextCleaner, EntityExtractor, Enricher, DatabaseLoader | SQLite rows |
| `classifier` | KeywordClassifier, MLClassifier, TransformerClassifier, DistilBertClassifier | `{category, confidence, scores}` |
| `analysis` | TrendAnalyzer, AnomalyDetector, ReportGenerator | Markdown/HTML reports |
| `export` | StixExporter, CsvExporter, MispExporter | STIX bundle, CSV, MISP JSON |
| `ai_summarizer` | ThreatSummarizer | Executive/technical/IOC bulletin text |
| `scheduler` | ToolkitScheduler | APScheduler jobs (scrape, classify, report) |
| `dashboard` | Streamlit app | Interactive web UI on port 8501 |